# Survey Monkey


# Configuration

The first code block ensures that Python is able to correctly locate the modules used for this pipeline

In [1]:
import sys
import os

sys.path.append(f'{os.getcwd()}/src')

In [2]:
surveyId = '314840976'

### Extract from Survey Monkey

Survey data is extracted from the Survey Monkey API with the raw data stored in JSON files in `/datalake`

In [3]:
from modules.survey_monkey import surveyMonkey

# surveyMonkey(surveyId)

### Transform data to normalized structured and load to database

This stage takes the extract from Survey Monkey, transforming it and loading normalized relational database structure

In [4]:
from modules.normalize import transformToNormalizedDb

transformToNormalizedDb(surveyId)

### Transform

This transformation stage provides domain-specific transformations to the data, combining it back into a single view.

These transformations include:
- Combine the count of followers across different social networks
- Calculate the current age based on date of birth

The data is then denormalised ready for loading and data exploration

In [5]:
from modules.survey_transforms.mfm_transforms import transformMfmTesters

transformedData = transformMfmTesters(surveyId)

12k
300 on personal. 
1.5k - my husband 
2000+ on mine
Hapgood:256
5.7k 
186
Looking to grow and post baby content
My partner has nearly 17,000
Rantzfam - 5,500
Benrantz - 152,000
10.6k (business)
prince_tyler2020 (728)
charmingseoul (781)
150.0k likes 
I have some videos with as much as 3.5M views 
iamgeorgiawray: 1133
4.6k on 1 account 
470 on another 

Supermummy secrets - over 5k

Lifestyle home & Christmas Inspired - over 8k
5.7k
Public-800

Instagram 940 tara_crichton
7.1k TheStyleRawr 
@TheStyleRawr - almost 2k

55.7k likes


### Load

The final load stage uses pandas to load a dataframe object into CSV, JSON and Excel file formats.

In [6]:
from modules.pandas import pandasLoad
from modules.survey_transforms.mfm_transforms import mfmColumns

dataColumns = mfmColumns(surveyId)

dataframe = pandasLoad(transformedData, dataColumns)

dataframe.head(10)



,What is the age of your youngest child?,What is the age of your next child (<strong>child 2</strong>)?,Aggregated Follower Count,What is the date of birth of your youngest child?,Which best describes your ethnic background?,Which sleep products is your baby or child/ren using ? Please select all that apply,How would you describe your family? Please select all that apply,Does your car have ISOFIX fittings?,How many children do you have?,Are you an influencer or blogger? Please select all that apply,Do you or will you use reusable nappies?,Which of the following best describe the ethnic group of <strong>your family</strong> (eg partner and/or children)? Please select all that apply,Do any members of your household have... Please select all that apply,Do any members of your household follow a specific diet? Please select all that apply,Please could you give us a few more details and who in your family experiences this,Please write a short review (no more than 100 words) about a pregnancy OR parenting product you've used. We're looking for your feedback on what you liked and didn't like about it. We may use this review to help us decide whether or not to choose you as a tester,What is your contact email?,"What is the name of your Facebook account? If possible, please do NOT include @ in your answer. Please note, we need this so we can recognise you when you request to join the MFM Top Testers Club",Have you or your partner used or are you planning to use a breast pump?,"What is your Instagram handle? If possible, please do NOT include @ in your answer. Please skip if you'd rather not say",Please tell us your first name,Do you have any pets living with you at home? Please select all that apply,How would you describe yourself...,"Does anyone in your household have food allergies, intolerances or sensitivities? Please select all that apply","I understand that if I am sent a product to test, I will provide MadeForMums with the required feedback within the specified deadline, or I will have to return the product(s) in their original and unused condition or be charged the RRP for it.",Do you have an Instagram account?,How important to you is choosing eco-friendly/sustainable parenting products?,Which of these best describes the area you live in?,"Are you currently using, or have you used in the past, a dummy or soother for your baby or child?",Do you or your family have any skincare conditions? Please select all that apply,What is your annual household income before tax?,Roughly how many followers do you have on your Facebook account? Please skip if you'd rather not say,Do you have any additional information you'd like to tell us about you or your children? (optional),Roughly how many followers do you have on your Instagram account? Please skip if you'd rather not say,Please can you tell us the make of your car,Please tell us your surname,What is the first name of your youngest child?,What is your mobile number? (This is so we can contact you if there are any queries about delivering a product to you or following up on feedback),What is your full postal address?,Do you have a TikTok account?,Do you have any children?,Does your baby or child suck their thumb?,"I understand and agree that if I am chosen to be a product tester, my name, postal address and phone number may be shared with the company who will be despatching the product to me (we will only pass this information on if it's essential for delivery)",Do you live in a:,Would you describe your youngest child as...,Do you have a Facebook account? Please note you will need a Facebook account in order to join the MFM Top Testers Club Facebook group,Do you have a Twitter account?,Do you have a YouTube channel?,Please tell us which age group you fall into,Which of the following best describe <strong>your</strong> ethnic group or background? This is just about you - we’ll ask about your family’s ethnic backgrounds in the next question,What is your current occupation? (If you're on mat l

In [ ]:
# Validation
from modules.metadata import getExtractMetadata
from modules.sqlSelects import getSurveyResponses

extractCount = getExtractMetadata(surveyId)['response_count']
normalizedLoadCount = len(getSurveyResponses(surveyId))
denormalizedLoadCount = dataframe.shape[0]

print(f'Total records in extract {extractCount}')
print(f'Total records in relational database {normalizedLoadCount}')
print(f'Total records in final load {denormalizedLoadCount}')

if extractCount == normalizedLoadCount == denormalizedLoadCount:
    print('Pipeline complete and validated')

Total records in extract 6170
Total records in relational database 6170
Total records in final load 6170
Pipeline complete and validated


In [ ]:
import pandas as pd

reachAgg = pd.DataFrame(dataframe['Aggregated Follower Count'])
reachAgg.sort_values(by=['Aggregated Follower Count'], ascending=False)

pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.DataFrame(dataframe['Aggregated Follower Count']).describe()

,Aggregated Follower Count
count,6170.00
mean,2354.79
std,49124.12
min,0.00
25%,0.00
50%,280.00
75%,967.00
max,2701870.00


In [9]:
from modules.survey_transforms.mfm_transforms import followerCountFromString

tests = [
    {
        'name': 'Basic String',
        'input': '10',
        'expected': 10
    },
    {
        'name': 'Integer',
        'input': 100,
        'expected': 100
    },
    {
        'name': '2.5k string',
        'input': '2.5k',
        'expected': 2500
    },
    {
        'name': '2.5K string',
        'input': '2.5K',
        'expected': 2500 
    },
    {
        'name': '10,000 string',
        'input': '10,000',
        'expected': 10000 
    },
    {
        'name': '1M string',
        'input': '1M',
        'expected': 1000000
    },
    {
        'name': '2.6m string',
        'input': '2.6m',
        'expected': 2600000
    },
    {
        'name': '1,000+ string',
        'input': '1,000+',
        'expected': 1000
    },
    {
        'name': 'Longer string',
        'input': 'Roughly 500',
        'expected': 500
    },
    {
        'name': 'Longer string with handle',
        'input': '@testaccount 10k',
        'expected': 10000
    },
    {
        'name': 'Over string',
        'input': 'Over 200000',
        'expected': 200000
    },
    {
        'name': 'Question string',
        'input': 'I think about 1000?',
        'expected': 1000
    }
]

print(followerCountFromString('''1. 1777
2. 2715
3. 2600
'''))

for test in tests:
    # Run transformation module
    # result = followerCountFromString(test['input'])
    # Check result is as expected
    isExpected = 'Yes ✅' if result == test['expected'] else 'No ⛔️'
    # Check result type is an integer
    isInt = 'Yes ✅' if type(result) == int else 'No ⛔️'

    # print(f'{test['name']}: {test['input']} became {result}. Expected result? {isExpected}. Is integer? {isInt}')
    

1


NameError: name 'result' is not defined

In [ ]:
from modules.survey_transforms.mfm_transforms import convertDobToAge

# Note: Dates are in US format mm/dd/yyyy!

tests = [
    {
        'input': '01/01/2024',
        'expected': 0
    },
    {
        'input': '06/05/2023',
        'expected': 1
    },
    {
        'input': '12/31/2020',
        'expected': 3
    },
    {
        'input': '09/05/2025',
        'expected': 0
    },
    {
        'input': '09/05/2030',
        'expected': -5
    },
    {
        'input': '09/10/1980',
        'expected': 44
    },
    {
        'input': '08/01/1998',
        'expected': 26
    }
]

for test in tests:
    # Run transformation module
    result = convertDobToAge(test['input'])
    # Check result is as expected
    isExpected = 'Yes ✅' if result == test['expected'] else 'No ⛔️'
    # Check result type is an integer or None
    isInt = 'Yes ✅' if type(result) == int or type(result) == type(None) else 'No ⛔️'

    print(f'{test['input']} became {result} years. Expected result? {isExpected}. Is integer or None? {isInt}')
    

01/01/2024 became 0 years. Expected result? Yes ✅. Is integer or None? Yes ✅
06/05/2023 became 1 years. Expected result? Yes ✅. Is integer or None? Yes ✅
12/31/2020 became 3 years. Expected result? Yes ✅. Is integer or None? Yes ✅
09/05/2025 became 0 years. Expected result? Yes ✅. Is integer or None? Yes ✅
09/05/2030 became -5 years. Expected result? Yes ✅. Is integer or None? Yes ✅
09/10/1980 became 44 years. Expected result? Yes ✅. Is integer or None? Yes ✅
08/01/1998 became 26 years. Expected result? Yes ✅. Is integer or None? Yes ✅
